# 13 — Pilot G0/G1/G2/G3, 20 epoch, EMA utama
Run ini hanya melatih generator dan menyiapkan review. Tidak melatih detektor atau membaca pixel/label test.
Input privat: texture_study_code.zip, melon-dataset-regrouped-v4, melon-benchmark-assets.
G0 kontrol; G1 nearest resize + conv; G2 ROI gray; G3 ROI RGB. Seluruh kondisi dari awal, seed11, kalibrasi warna train-only, patch32/stride16/count4/coverage0.95/lambda0.5.
Batas kernel di CLI Kaggle: 110 menit; tiap generator maksimal30menit. Run parsial gagal tertutup.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/texture_study_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json') if (p.parent/'code/cyclegan/tools/texture_study.py').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
if not torch.cuda.is_available(): raise RuntimeError('GPU required for pilot')
run('-m','unittest','discover','-s',HERE/'tests','-p','test_texture*.py')
SOURCE=WORK/'source_crops'
POOL=WORK/'curated_crops'
run(HERE/'tools/build_crops.py','--dataset',DATASET,'--out',SOURCE)
run(HERE/'tools/prepare_g0x_data.py','--source',SOURCE,'--out',POOL)
STUDY=WORK/'study'
run(CLI,'prepare','--study',STUDY,'--dataset',DATASET,'--crops',POOL)


In [ ]:
run(CLI,'train','--study',STUDY,'--fraction',100,'--arms','G0','G1','G2','G3','--budget-minutes',30)
initial={a:json.loads((STUDY/f'p100/checkpoints/p100_{a}_seed11/initial_weights.json').read_text()) for a in ['G0','G1','G2','G3']}
assert initial['G0']==initial['G2']==initial['G3'], 'unmatched shared initialization'
assert all(initial['G1'][k]==initial['G0'][k] for k in ['D_A','D_B'])


In [ ]:
run(CLI,'export','--study',STUDY,'--dataset',DATASET,'--fraction',100,'--device','cuda')
run(CLI,'diagnose','--study',STUDY,'--fraction',100,'--device','cuda')
run(CLI,'review','--study',STUDY,'--fraction',100)


In [ ]:
from IPython.display import display, Image
for arm in ['G0','G1','G2','G3']:
    for weights in ['ema','raw']:
        out=STUDY/'p100/panels'/f'{arm}_{weights}'
        run(HERE/'tools/texture_panel.py','--crops',STUDY/'p100/crops','--checkpoints-dir',STUDY/'p100/checkpoints',
            '--name',f'p100_{arm}_seed11','--weights',weights,'--out',out,'--device','cuda','--patch-size',32)
        print(arm,weights);display(Image(filename=str(out/'panel_00.png')))
print('REVIEW REQUIRED:',STUDY/'p100/review/index.html')
print('Provide only the review folder to the reviewer; keep review_key_PRIVATE.json hidden.')


## Batas tahap
Isi review.csv dengan keputusan manusia (yes/no dan reviewer). File kosong sengaja tidak diterima.
Jalankan `select` hanya setelah review lengkap. Jangan menentukan acceptance dari skor diagnostik.
Kaggle output memuat key privat; distribusikan hanya folder review kepada reviewer. EMA adalah hasil primer, raw diagnostik.
Biaya GPU dicatat dari perubahan quota API di ledger host; training walltime tersimpan per arm.
